# Proyecto Empresa Aliada 4 — Análisis exploratorio de ventas
#### Objetivo: consolidar las cinco tablas de la actividad anterior, revisar sus variables, crear visualizaciones y registrar hallazgos para el dashboard final.
##### //Nota sobre los datos: TOTAL_VALUE_SALES es el valor de ventas reportado por la fuente, pero no se especifica aquí una moneda. Además, TOTAL AUTOS SCANNING MEXICO es un total nacional, no una séptima zona: se analiza por separado de las áreas 1–6 para no duplicar ventas. Se tienen 52 semanas de 2022 y 28 de 2023, por lo que ambos años no deben compararse directamente sin ajustar el periodo.
##### //El código utiliza pasos sencillos similares a los vistos en clase. Cada gráfico se guarda como PNG en Downloads/Proyecto_EA4.

## 1. Importar librerías y preparar la carpeta
#### Usaremos pandas para manipular tablas, matplotlib y seaborn para los gráficos. Si falta alguna librería, se puede instalar con %pip install pandas openpyxl matplotlib seaborn numpy.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
# Portfolio adaptation: portable paths; original analytical cells unchanged.
candidatos = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in candidatos if (p / "data" / "python").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")
carpeta_datos = raiz / "data" / "python"
carpeta_salida = raiz / "reports" / "generated"
carpeta_salida.mkdir(parents=True, exist_ok=True)
def buscar_archivo(nombre):
    for ruta in [carpeta_datos / nombre, carpeta_datos / (nombre + ".gz")]:
        if ruta.exists():
            return ruta
    raise FileNotFoundError("Missing input: " + nombre)
print("Gráficas y resultados en:", carpeta_salida)


## 2. Cargar y revisar los archivos originales
#### La tabla FACT_SALES contiene las ventas; las demás tablas aportan características de los productos, segmentos, categorías y calendario.

In [ ]:
# Cargamos los 5 archivos del proyecto.
df_category = pd.read_csv(buscar_archivo("DIM_CATEGORY.csv"))
df_product = pd.read_excel(buscar_archivo("DIM_PRODUCT.xlsx"), dtype={"ITEM": "string"})
df_segment = pd.read_excel(buscar_archivo("DIM_SEGMENT.xlsx"))
df_calendar = pd.read_excel(buscar_archivo("DIM_CALENDAR.xlsx"), dtype={"WEEK": "string"})
df_sales = pd.read_csv(buscar_archivo("FACT_SALES.csv"), dtype={"WEEK": "string", "ITEM_CODE": "string"})

for nombre, tabla in {"Ventas": df_sales, "Productos": df_product,
                      "Calendario": df_calendar, "Categorías": df_category,
                      "Segmentos": df_segment}.items():
    print(nombre, "-", tabla.shape[0], "filas y", tabla.shape[1], "columnas")

print("\nPrimeras filas de ventas:")
print(df_sales.head())
print("\nTipos de datos:")
print(df_sales.dtypes)

## 3. Limpiar y consolidar las tablas
#### Se normalizan espacios, nombres y tipos; después se relacionan las claves con merge(). En el archivo previo (Proyecto_EA3) ya se revisaban las tablas y se limpiaban textos: retomamos ese procedimiento y completamos las uniones.

In [ ]:
# Limpieza básica: quita espacios, saltos de línea y unifica textos.
def limpiar_textos(tabla):
    tabla = tabla.copy()
    for columna in tabla.select_dtypes(include=["object", "string"]).columns:
        tabla[columna] = (tabla[columna].astype("string")
                          .str.replace(r"\s+", " ", regex=True)
                          .str.strip().str.upper())
    return tabla

df_category = limpiar_textos(df_category).drop_duplicates()
df_product = limpiar_textos(df_product).drop_duplicates()
df_segment = limpiar_textos(df_segment).drop_duplicates()
df_calendar = limpiar_textos(df_calendar).drop_duplicates()
df_sales = limpiar_textos(df_sales).drop_duplicates()

# No se deben repetir claves en tablas de consulta.
df_product = df_product.dropna(subset=["ITEM"]).drop_duplicates("ITEM")
df_calendar = df_calendar.drop_duplicates("WEEK")
df_category = df_category.drop_duplicates("ID_CATEGORY")
claves_segmento = ["CATEGORY", "ATTR1", "ATTR2", "ATTR3", "FORMAT"]
df_segment = df_segment.drop_duplicates(claves_segmento)

for columna in ["TOTAL_VALUE_SALES", "TOTAL_UNIT_SALES", "TOTAL_UNIT_AVG_WEEKLY_SALES"]:
    df_sales[columna] = pd.to_numeric(df_sales[columna], errors="coerce")
df_calendar["DATE"] = pd.to_datetime(df_calendar["DATE"], errors="coerce")

# Cruce de ventas con calendario y productos.
datos = df_sales.merge(df_calendar, on="WEEK", how="left", validate="many_to_one")
datos = datos.merge(df_product, left_on="ITEM_CODE", right_on="ITEM", how="left", validate="many_to_one")

# Nombre de categoría y segmento.
df_category = df_category.rename(columns={"CATEGORY": "CATEGORY_NAME"})
datos = datos.merge(df_category, left_on="CATEGORY", right_on="ID_CATEGORY",
                    how="left", validate="many_to_one")
datos = datos.merge(df_segment[claves_segmento + ["SEGMENT"]],
                    on=claves_segmento, how="left", validate="many_to_one")

print("Filas consolidadas:", len(datos))
print("Fechas:", datos["DATE"].min(), "a", datos["DATE"].max())
print("Registros sin marca:", datos["BRAND"].isna().sum())
print("Registros sin segmento:", datos["SEGMENT"].isna().sum())
print("Categorías distintas:", datos["CATEGORY_NAME"].nunique())

# Se guarda el dataset consolidado para futuras actividades.
datos.to_csv(carpeta_salida / "ventas_consolidadas_EA4.csv", index=False)

## 4. Identificar variables y separar nacional de regional
#### Para responder preguntas de negocio se utilizarán: TOTAL_VALUE_SALES, TOTAL_UNIT_SALES, DATE, YEAR, BRAND, SEGMENT y REGION. Como la información disponible pertenece a una sola categoría principal, comparar marcas y segmentos es más útil que comparar categorías.

In [ ]:
# Separamos el total México de las seis áreas.
ventas_nacional = datos[datos["REGION"] == "TOTAL AUTOS SCANNING MEXICO"].copy()
ventas_areas = datos[datos["REGION"] != "TOTAL AUTOS SCANNING MEXICO"].copy()

print("Registros de ventas nacionales:", len(ventas_nacional))
print("Registros de las áreas:", len(ventas_areas))
print("Regiones disponibles:", ventas_areas["REGION"].nunique())
print("\nSemanas registradas por año en México:")
print(ventas_nacional.groupby("YEAR")["WEEK"].nunique())
print("\nEstadísticas del valor nacional de ventas:")
print(ventas_nacional["TOTAL_VALUE_SALES"].describe())

# Una muestra del consolidado.
print("\nMuestra de datos consolidados:")
print(datos[["DATE", "BRAND", "SEGMENT", "REGION",
             "TOTAL_UNIT_SALES", "TOTAL_VALUE_SALES"]].sample(5, random_state=42))

## 5. Distribución de valores de venta: histograma
#### Pregunta: ¿la mayoría de los registros tienen ventas pequeñas o grandes? Se usa hasta el percentil 99 para que los valores extremos no oculten la forma general; esos extremos se revisarán por separado.

In [ ]:
limite_99 = ventas_nacional["TOTAL_VALUE_SALES"].quantile(0.99)
ventas_hist = ventas_nacional[ventas_nacional["TOTAL_VALUE_SALES"] <= limite_99]

plt.figure(figsize=(10, 5))
sns.histplot(ventas_hist["TOTAL_VALUE_SALES"], bins=45, color="#219E9A")
plt.title("Distribución del valor de ventas por registro (México)")
plt.xlabel("Valor de ventas reportado")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.savefig(carpeta_salida / "01_histograma_ventas.png", dpi=160)
plt.show()
plt.close()
print("Mediana:", round(ventas_nacional["TOTAL_VALUE_SALES"].median(), 2))
print("Percentil 99:", round(limite_99, 2))

#### Hallazgo 1: La distribución se concentra en valores pequeños, con mediana de 52.59 por registro nacional; el percentil 99 llega a 4,105.90, lo que confirma una cola de valores elevados. En el histograma se recorta solo la visualización, no el dataset.

## 6. Distribución por segmento: boxplot
#### Pregunta: ¿qué segmentos presentan ventas típicas más altas o mayor dispersión? Usamos log1p únicamente para visualizar mejor la gran diferencia entre registros; el dato original de ventas no cambia.

In [ ]:
ventas_nacional["VALOR_LOG"] = np.log1p(ventas_nacional["TOTAL_VALUE_SALES"])
orden_segmentos = (ventas_nacional.groupby("SEGMENT")["TOTAL_VALUE_SALES"]
                   .median().sort_values(ascending=False).index)
plt.figure(figsize=(11, 5))
sns.boxplot(data=ventas_nacional, x="SEGMENT", y="VALOR_LOG",
            order=orden_segmentos, color="#73B4D2", fliersize=1)
plt.xticks(rotation=25, ha="right")
plt.title("Distribución de ventas por segmento (escala log1p)")
plt.xlabel("Segmento")
plt.ylabel("log(1 + valor de ventas)")
plt.tight_layout()
plt.savefig(carpeta_salida / "02_boxplot_segmentos.png", dpi=160)
plt.show()
plt.close()

#### Hallazgo 2: Los segmentos POWDER y BAR tienen medianas por registro de 106.59 y 102.07, respectivamente; SANITIZER tiene una mediana de 25.04. Esto no significa que POWDER sea el segmento con más ventas totales.

## 7. Tendencia de las ventas nacionales por semana
#### Pregunta: ¿cómo cambian las ventas a lo largo de los periodos observados? Esta gráfica mantiene las fechas reales y no supone que 2023 esté completo.

In [ ]:
ventas_semanales = (ventas_nacional.groupby("DATE", as_index=False)["TOTAL_VALUE_SALES"]
                   .sum().sort_values("DATE"))
plt.figure(figsize=(11, 5))
plt.plot(ventas_semanales["DATE"], ventas_semanales["TOTAL_VALUE_SALES"],
         color="#087F8C", linewidth=2)
plt.title("Evolución semanal de las ventas nacionales")
plt.xlabel("Fecha")
plt.ylabel("Valor total de ventas")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(carpeta_salida / "03_tendencia_semanal.png", dpi=160)
plt.show()
plt.close()

#### Hallazgo 3: Las ventas semanales varían; la mayor semana nacional observada es la del 5 de junio de 2023 con 99,891.76 en valor de ventas. La base termina en julio de 2023: no representa el año completo.

## 8. Tendencia de ventas mensuales por marca
#### Pregunta: ¿cuáles marcas impulsan el mercado y cómo se comportan en el tiempo? Elegimos las cuatro marcas con mayor valor de ventas en México.

In [ ]:
marcas_top = (ventas_nacional.groupby("BRAND")["TOTAL_VALUE_SALES"]
              .sum().nlargest(4).index)
ventas_marca = ventas_nacional[ventas_nacional["BRAND"].isin(marcas_top)].copy()
ventas_marca["MES"] = ventas_marca["DATE"].dt.to_period("M").dt.to_timestamp()
serie_marca = (ventas_marca.groupby(["MES", "BRAND"])["TOTAL_VALUE_SALES"]
               .sum().unstack(fill_value=0))
plt.figure(figsize=(11, 5))
for marca in serie_marca.columns:
    plt.plot(serie_marca.index, serie_marca[marca], marker="o", markersize=3, label=marca)
plt.title("Ventas mensuales de las principales marcas")
plt.xlabel("Mes")
plt.ylabel("Valor de ventas")
plt.xticks(rotation=30)
plt.legend(title="Marca")
plt.tight_layout()
plt.savefig(carpeta_salida / "04_marcas_tiempo.png", dpi=160)
plt.show()
plt.close()

#### Hallazgo 4: CLORALEX es la marca de mayor valor de ventas acumulado (aprox. 2.70 millones en las unidades reportadas). Se observa que su línea permanece sobre la de las otras marcas principales durante los meses de la muestra.

## 9. Relación entre volumen y valor de ventas: dispersión
#### Pregunta: ¿los registros con más unidades también suelen tener un valor mayor? La correlación de Pearson indica el grado de relación lineal, pero no demuestra causalidad. Visualizamos una muestra de 3,000 registros para que los puntos no tapen el gráfico.

In [ ]:
muestra = ventas_nacional.sample(n=min(3000, len(ventas_nacional)), random_state=42)
correlacion = ventas_nacional["TOTAL_UNIT_SALES"].corr(ventas_nacional["TOTAL_VALUE_SALES"])
plt.figure(figsize=(9, 5))
sns.scatterplot(data=muestra, x="TOTAL_UNIT_SALES", y="TOTAL_VALUE_SALES",
                alpha=0.35, color="#12658A", s=26)
plt.title("Unidades vendidas vs. valor de ventas")
plt.xlabel("Unidades vendidas (valor reportado)")
plt.ylabel("Valor de ventas")
plt.tight_layout()
plt.savefig(carpeta_salida / "05_scatter_unidades_valor.png", dpi=160)
plt.show()
plt.close()
print("Correlación de Pearson:", round(correlacion, 3))

#### Hallazgo 5: Existe una relación positiva fuerte entre unidades y valor vendido (r = 0.914). Los puntos más alejados pueden estar asociados a distintos precios, presentaciones o tamaños de compra; la gráfica por sí sola no determina la causa.

## 10. Distribución geográfica: barras apiladas por área y segmento
#### Pregunta: ¿qué áreas tienen mayor valor de ventas y qué segmentos contribuyen más? Se excluye la fila agregada TOTAL AUTOS SCANNING MEXICO para no duplicar el total.

In [ ]:
tabla_areas = (ventas_areas.groupby(["REGION", "SEGMENT"])["TOTAL_VALUE_SALES"]
               .sum().unstack(fill_value=0))
tabla_areas = tabla_areas.loc[tabla_areas.sum(axis=1).sort_values(ascending=False).index]
tabla_areas.index = tabla_areas.index.str.replace("TOTAL AUTOS ", "", regex=False)
ax = tabla_areas.plot(kind="bar", stacked=True, figsize=(11, 6), colormap="tab20c")
ax.set_title("Valor de ventas por área y segmento")
ax.set_xlabel("Área")
ax.set_ylabel("Valor de ventas")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Segmento", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(carpeta_salida / "06_areas_segmentos.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

#### Hallazgo 6: AREA 2 tiene el mayor valor acumulado (1,188,796.15), mientras que AREA 4 presenta el menor (677,436.00). La gráfica usa solo las áreas, sin agregar otra vez el total nacional.

## 11. Revisar outliers de ventas por marca
#### Un outlier es un dato muy alejado del comportamiento usual. En esta práctica se identifican valores por arriba de Q3 + 1.5 × IQR usando los registros nacionales; esto es una señal para investigar, no significa que el dato sea incorrecto.

In [ ]:
marcas_6 = (ventas_nacional.groupby("BRAND")["TOTAL_VALUE_SALES"].sum()
            .nlargest(6).index)
df_outliers = ventas_nacional[ventas_nacional["BRAND"].isin(marcas_6)].copy()

# Criterio IQR dentro de cada marca.
resumen_outliers = []
for marca, tabla in df_outliers.groupby("BRAND"):
    q1 = tabla["TOTAL_VALUE_SALES"].quantile(.25)
    q3 = tabla["TOTAL_VALUE_SALES"].quantile(.75)
    limite = q3 + 1.5 * (q3 - q1)
    extremos = (tabla["TOTAL_VALUE_SALES"] > limite).sum()
    resumen_outliers.append({"BRAND": marca, "Límite IQR": limite,
                             "Registros por revisar": extremos})
print(pd.DataFrame(resumen_outliers).sort_values("Registros por revisar", ascending=False))

df_outliers["VALOR_LOG"] = np.log1p(df_outliers["TOTAL_VALUE_SALES"])
plt.figure(figsize=(11, 5))
sns.boxplot(data=df_outliers, x="BRAND", y="VALOR_LOG", color="#85B8D0", fliersize=1)
plt.title("Valores extremos de ventas por marca (escala log1p)")
plt.xlabel("Marca")
plt.ylabel("log(1 + valor de ventas)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(carpeta_salida / "07_outliers_marcas.png", dpi=160)
plt.show()
plt.close()

#### Hallazgo 7: Hay observaciones alejadas de la distribución habitual de varias marcas. Conviene revisar los registros marcados por IQR, pero un outlier no es automáticamente un error; puede representar ventas realmente altas.

## 12. Oportunidad: participación de marcas
#### Pregunta: ¿qué tan concentradas están las ventas? El porcentaje se calcula frente al total nacional, no frente a la suma duplicada de áreas.

In [ ]:
marca_valor = (ventas_nacional.groupby("BRAND")["TOTAL_VALUE_SALES"]
               .sum().sort_values(ascending=False))
participacion = 100 * marca_valor / marca_valor.sum()
plt.figure(figsize=(10, 5))
sns.barplot(x=participacion.head(10).values, y=participacion.head(10).index, color="#168F94")
plt.title("Participación de las principales marcas en ventas nacionales")
plt.xlabel("Porcentaje del valor de ventas nacional (%)")
plt.ylabel("Marca")
plt.tight_layout()
plt.savefig(carpeta_salida / "08_participacion_marcas.png", dpi=160)
plt.show()
plt.close()
print("Marca principal:", participacion.index[0], "-", round(participacion.iloc[0], 1), "%")

#### Hallazgo 8: CLORALEX concentra 48.8% del valor nacional de ventas del periodo. La concentración por marca podría orientar qué competidores o productos conviene estudiar con más detalle.

## 13. Oportunidad: comparar 2022 y 2023 con semanas equivalentes
#### Para no confundir meses faltantes con caída de ventas, se comparan solamente los números de semana presentes en ambos años. Esto es más justo que comparar el total de 2022 con el de un 2023 incompleto.

In [ ]:
semanas_2022 = set(ventas_nacional.loc[ventas_nacional["YEAR"] == 2022, "WEEK_NUMBER"])
semanas_2023 = set(ventas_nacional.loc[ventas_nacional["YEAR"] == 2023, "WEEK_NUMBER"])
semanas_comunes = semanas_2022.intersection(semanas_2023)
ventas_comparables = ventas_nacional[
    ventas_nacional["YEAR"].isin([2022, 2023]) &
    ventas_nacional["WEEK_NUMBER"].isin(semanas_comunes)
]
comparacion = (ventas_comparables.groupby(["SEGMENT", "YEAR"])["TOTAL_VALUE_SALES"]
               .sum().unstack(fill_value=0))
comparacion["CRECIMIENTO_%"] = np.where(
    comparacion[2022] > 0,
    (comparacion[2023] / comparacion[2022] - 1) * 100,
    np.nan
)
print("Semanas comparadas por año:", len(semanas_comunes))
print(comparacion.sort_values("CRECIMIENTO_%", ascending=False).round(2))

comparacion[[2022, 2023]].sort_values(2023, ascending=False).plot(
    kind="bar", figsize=(11, 5), color=["#4A82B2", "#17A49C"]
)
plt.title("Ventas por segmento: mismas semanas de 2022 y 2023")
plt.xlabel("Segmento")
plt.ylabel("Valor de ventas")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Año")
plt.tight_layout()
plt.savefig(carpeta_salida / "09_comparacion_periodos.png", dpi=160)
plt.show()
plt.close()

#### Hallazgo 9: Comparando las primeras 28 semanas de 2022 y 2023, PRETREAT aumenta 23.6% y SANITIZER disminuye 30.0%. Es una comparación de periodos equivalentes, no de años completos.

## 14. Hallazgos principales y archivos de salida
#### La siguiente celda imprime hallazgos calculados a partir de los datos, y los guarda como texto para incluirlos después en el dashboard o la presentación. Los valores altos no implican necesariamente un error: conviene revisarlos por contexto de negocio.

In [ ]:
area_totales = (ventas_areas.groupby("REGION")["TOTAL_VALUE_SALES"].sum()
               .sort_values(ascending=False))
segmento_totales = (ventas_nacional.groupby("SEGMENT")["TOTAL_VALUE_SALES"].sum()
                    .sort_values(ascending=False))
mediana = ventas_nacional["TOTAL_VALUE_SALES"].median()
max_crecimiento = comparacion["CRECIMIENTO_%"].idxmax()
min_crecimiento = comparacion["CRECIMIENTO_%"].idxmin()

hallazgos = [
    "PROYECTO EA4 - HALLAZGOS PRINCIPALES",
    "Fuente: FACT_SALES + DIM_PRODUCT + DIM_CATEGORY + DIM_SEGMENT + DIM_CALENDAR",
    "1. Registros consolidados: " + str(len(datos)) + ".",
    "2. Periodo con datos: " + str(datos["DATE"].min().date()) + " a " + str(datos["DATE"].max().date()) + ".",
    "3. El valor mediano por registro nacional es " + f"{mediana:,.2f}" + ".",
    "4. El segmento con mayor valor nacional es " + str(segmento_totales.index[0]) + ".",
    "5. El área con mayor valor de ventas es " + str(area_totales.index[0]) + ".",
    "6. La correlación entre unidades y valor de ventas es " + f"{correlacion:.3f}" + ".",
    "7. La marca con mayor participación es " + str(participacion.index[0]) +
        " (" + f"{participacion.iloc[0]:.1f}%" + ").",
    "8. Comparando " + str(len(semanas_comunes)) + " semanas equivalentes, el segmento con mayor variación es " +
        str(max_crecimiento) + " (" + f"{comparacion.loc[max_crecimiento, 'CRECIMIENTO_%']:.1f}%" + ").",
    "9. En ese mismo periodo, el segmento con menor variación es " +
        str(min_crecimiento) + " (" + f"{comparacion.loc[min_crecimiento, 'CRECIMIENTO_%']:.1f}%" + ").",
    "10. Solo hay una categoría principal. Comparar segmentos, marcas y áreas es más informativo.",
    "IMPORTANTE: el total nacional y las seis áreas no se suman entre sí.",
    "IMPORTANTE: 2023 es parcial; se deben comparar periodos equivalentes.",
    "IMPORTANTE: no se declara moneda para TOTAL_VALUE_SALES en el dataset.",
]

for linea in hallazgos:
    print(linea)
(carpeta_salida / "Insights_Proyecto_EA4.txt").write_text("\n".join(hallazgos), encoding="utf-8")
comparacion.reset_index().to_csv(carpeta_salida / "comparacion_segmentos_2022_2023.csv", index=False)
print("\nArchivos PNG y reportes guardados en:", carpeta_salida)